# Notebook 03 — Evaluate All Experiments

**Member responsible: C**

Runs evaluation for every experiment and saves results CSVs.
Run once per experiment by updating `configs/eval.yaml`.

| Experiment | `checkpoint_path` in eval.yaml | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                                  | GSM8K + ARC |
| Exp 1 few-shot CoT | `null`                                  | GSM8K + ARC |
| Exp 2 adaptation   | `results/checkpoints/adaptation`        | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation`      | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`          | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered` (same)  | ARC only |

In [ ]:
# Step 1a: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 1b: Clone repo
# TODO: replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/SC4001-Group133.git /content/SC4001-Group133
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 1c: Redirect HF cache + data/ + results/ to Drive
# MUST run before any HuggingFace imports
# Checkpoints saved by notebook 02 are automatically available via results/ symlink
# Datasets cached from previous runs are reused — no re-downloading
import sys
sys.path.insert(0, '/content/SC4001-Group133')
from src.setup_cache import setup
setup()

In [ ]:
# Step 2: Run Exp 0 — Zero-shot baseline (no checkpoint needed)
from src.baselines.cot_prompting import run_baseline
from src.evaluation.metrics import compute_summary_stats, save_results

results_exp0 = run_baseline(mode='zero_shot', dataset_name='gsm8k')
save_results(results_exp0, 'results/exp0_zero_shot_gsm8k.csv')
print('Exp 0 GSM8K:', compute_summary_stats(results_exp0))

In [ ]:
# Step 3: Run Exp 1 — Few-shot CoT baseline
results_exp1 = run_baseline(mode='few_shot', dataset_name='gsm8k')
save_results(results_exp1, 'results/exp1_few_shot_gsm8k.csv')
print('Exp 1 GSM8K:', compute_summary_stats(results_exp1))

In [ ]:
# Step 4: Run Exp 2-4 — update eval.yaml checkpoint_path, then run this cell
# TODO: edit configs/eval.yaml:
#   model.checkpoint_path: results/checkpoints/adaptation   (for Exp 2)
#   model.checkpoint_path: results/checkpoints/distillation (for Exp 3)
#   model.checkpoint_path: results/checkpoints/filtered     (for Exp 4)
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # TODO: change this per run

from src.evaluation.eval_gsm8k import evaluate_gsm8k
results = evaluate_gsm8k(experiment_name=EXPERIMENT_NAME)
print(f'{EXPERIMENT_NAME} done.')

In [ ]:
# Step 5: Run Exp 5 (D3) — ARC cross-task transfer
# Uses the Exp 4 checkpoint — make sure eval.yaml points to results/checkpoints/filtered
from src.evaluation.eval_arc import evaluate_arc
results_arc = evaluate_arc(experiment_name='exp5_d3_cross_task')
print('Exp 5 ARC done.')

In [ ]:
# Step 6: Confirm all results are on Drive
import os
results_drive = '/content/drive/MyDrive/SC4001/results'
csvs = [f for f in os.listdir(results_drive) if f.endswith('.csv')]
print(f'CSVs on Drive ({len(csvs)}):')
for f in sorted(csvs):
    print(f'  {f}')